In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from torch.utils.data import Dataset, DataLoader


def load_data(file_name):
    df = pd.read_csv(file_name)
    df.dropna(inplace=True)
    return df


bitcoin_df = load_data("bitcoin_processed.csv")
ethereum_df = load_data("ethereum_processed.csv")

In [2]:
scaler = MinMaxScaler()

def preprocess_data(df):
    features = ['open', 'high', 'low', 'volume', 'quote_asset_volume',
                'number_of_trades', 'taker_buy_base_asset_volume',
                'taker_buy_quote_asset_volume', 'average_price', 'price_change']
    target = 'target_close'
    
    df[features] = scaler.fit_transform(df[features])
    df[target] = scaler.fit_transform(df[[target]])
    return df, features, target

bitcoin_df, features, target = preprocess_data(bitcoin_df)
ethereum_df, _, _ = preprocess_data(ethereum_df)

In [3]:
def split_data(df, split_ratio=0.8):
    split_index = int(len(df) * split_ratio)
    train_data = df[:split_index]
    test_data = df[split_index:]
    return train_data, test_data

btc_train, btc_test = split_data(bitcoin_df)
eth_train, eth_test = split_data(ethereum_df)

In [4]:
def create_sequences(data, features, target, seq_length=10):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[features].iloc[i:i+seq_length].values)
        y.append(data[target].iloc[i+seq_length])
    return np.array(X), np.array(y)

SEQ_LENGTH = 10
X_btc_train, y_btc_train = create_sequences(btc_train, features, target, SEQ_LENGTH)
X_btc_test, y_btc_test = create_sequences(btc_test, features, target, SEQ_LENGTH)
X_eth_train, y_eth_train = create_sequences(eth_train, features, target, SEQ_LENGTH)
X_eth_test, y_eth_test = create_sequences(eth_test, features, target, SEQ_LENGTH)

In [5]:
X_btc_train, y_btc_train = torch.tensor(X_btc_train, dtype=torch.float32), torch.tensor(y_btc_train, dtype=torch.float32)
X_btc_test, y_btc_test = torch.tensor(X_btc_test, dtype=torch.float32), torch.tensor(y_btc_test, dtype=torch.float32)
X_eth_train, y_eth_train = torch.tensor(X_eth_train, dtype=torch.float32), torch.tensor(y_eth_train, dtype=torch.float32)
X_eth_test, y_eth_test = torch.tensor(X_eth_test, dtype=torch.float32), torch.tensor(y_eth_test, dtype=torch.float32)

In [6]:
class CryptoDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 64
btc_train_loader = DataLoader(CryptoDataset(X_btc_train, y_btc_train), batch_size=batch_size, shuffle=False)
btc_test_loader = DataLoader(CryptoDataset(X_btc_test, y_btc_test), batch_size=batch_size, shuffle=False)
eth_train_loader = DataLoader(CryptoDataset(X_eth_train, y_eth_train), batch_size=batch_size, shuffle=False)
eth_test_loader = DataLoader(CryptoDataset(X_eth_test, y_eth_test), batch_size=batch_size, shuffle=False)

In [7]:
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(RNNModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_size = len(features)
hidden_size = 64
num_layers = 10
output_size = 1
learning_rate = 0.001
num_epochs = 500


In [9]:
def train_model(model, train_loader, criterion, optimizer):
    model.train()
    for epoch in range(num_epochs):
        epoch_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch).squeeze()
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        if epoch % 10 == 0:
            print(f'Epoch [{epoch}/{num_epochs}], Loss: {epoch_loss/len(train_loader):.6f}')


In [10]:
btc_model = RNNModel(input_size, hidden_size, num_layers, output_size).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(btc_model.parameters(), lr=learning_rate)
train_model(btc_model, btc_train_loader, criterion, optimizer)

eth_model = RNNModel(input_size, hidden_size, num_layers, output_size).to(device)
optimizer = optim.Adam(eth_model.parameters(), lr=learning_rate)
train_model(eth_model, eth_train_loader, criterion, optimizer)

Epoch [0/500], Loss: 0.022532
Epoch [10/500], Loss: 0.054542
Epoch [20/500], Loss: 0.042283
Epoch [30/500], Loss: 0.041664
Epoch [40/500], Loss: 0.012882
Epoch [50/500], Loss: 0.003204
Epoch [60/500], Loss: 0.013847
Epoch [70/500], Loss: 0.002592
Epoch [80/500], Loss: 0.002040
Epoch [90/500], Loss: 0.001595
Epoch [100/500], Loss: 0.001588
Epoch [110/500], Loss: 0.001613
Epoch [120/500], Loss: 0.001648
Epoch [130/500], Loss: 0.008656
Epoch [140/500], Loss: 0.027550
Epoch [150/500], Loss: 0.004677
Epoch [160/500], Loss: 0.002426
Epoch [170/500], Loss: 0.001928
Epoch [180/500], Loss: 0.001695
Epoch [190/500], Loss: 0.001954
Epoch [200/500], Loss: 0.002514
Epoch [210/500], Loss: 0.001642
Epoch [220/500], Loss: 0.001580
Epoch [230/500], Loss: 0.001582
Epoch [240/500], Loss: 0.001831
Epoch [250/500], Loss: 0.006702
Epoch [260/500], Loss: 0.014615
Epoch [270/500], Loss: 0.003591
Epoch [280/500], Loss: 0.002188
Epoch [290/500], Loss: 0.001848
Epoch [300/500], Loss: 0.001839
Epoch [310/500], Lo

In [11]:
def evaluate_model(model, X, y):
    model.eval()
    with torch.no_grad():
        X, y = X.to(device), y.to(device)
        predictions = model(X).squeeze().cpu().numpy()
        y_true = y.cpu().numpy()
        mse = mean_squared_error(y_true, predictions)
        mae = mean_absolute_error(y_true, predictions)
        rmse = np.sqrt(mse)
        return mse, mae, rmse

mse_btc, mae_btc, rmse_btc = evaluate_model(btc_model, X_btc_test, y_btc_test)
print(f'Bitcoin - MSE: {mse_btc:.6f}, MAE: {mae_btc:.6f}, RMSE: {rmse_btc:.6f}')

mse_eth, mae_eth, rmse_eth = evaluate_model(eth_model, X_eth_test, y_eth_test)
print(f'Ethereum - MSE: {mse_eth:.6f}, MAE: {mae_eth:.6f}, RMSE: {rmse_eth:.6f}')


Bitcoin - MSE: 0.076566, MAE: 0.225427, RMSE: 0.276705
Ethereum - MSE: 0.007523, MAE: 0.065720, RMSE: 0.086736


In [ ]:
def evaluate_model_with_inverse_scaling(model, X, y, scaler):
    model.eval()
    with torch.no_grad():
        X, y = X.to(device), y.to(device)  
        
        predictions = model(X).squeeze().cpu().numpy()
        y_true = y.cpu().numpy()

        y_true = y_true.reshape(-1, 1)
        predictions = predictions.reshape(-1, 1)

        y_true_actual = scaler.inverse_transform(y_true)
        predictions_actual = scaler.inverse_transform(predictions)


        mse = mean_squared_error(y_true_actual, predictions_actual)
        mae = mean_absolute_error(y_true_actual, predictions_actual)
        rmse = np.sqrt(mse)

        return mse, mae, rmse, y_true_actual, predictions_actual

In [23]:
mse_btc_inv, mae_btc_inv, rmse_btc_inv, y_true_btc_inv, y_pred_btc_inv = evaluate_model_with_inverse_scaling(
    btc_model, X_btc_test, y_btc_test, scaler
)
print(f'Bitcoin (Inverse Scaled) - MSE: {mse_btc_inv:.6f}, MAE: {mae_btc_inv:.6f}, RMSE: {rmse_btc_inv:.6f}')

mse_eth_inv, mae_eth_inv, rmse_eth_inv, y_true_eth_inv, y_pred_eth_inv = evaluate_model_with_inverse_scaling(
    eth_model, X_eth_test, y_eth_test, scaler
)
print(f'Ethereum (Inverse Scaled) - MSE: {mse_eth_inv:.6f}, MAE: {mae_eth_inv:.6f}, RMSE: {rmse_eth_inv:.6f}')


Bitcoin (Inverse Scaled) - MSE: 2193002.500000, MAE: 1368.554688, RMSE: 1480.878962
Ethereum (Inverse Scaled) - MSE: 66006.468750, MAE: 194.666031, RMSE: 256.917241
